# Evaluation
Prioritizing: 
1. Macro Pearson correlation
2. Macro RMSE
3. Group-level behavior
4. Consistency across seeds

In [3]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [5]:
from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/2026-1/NLP/appraisal/nlp-appraisal"
)

RUNS_ROOT = PROJECT_ROOT / "runs"
DATA_ROOT = PROJECT_ROOT / "data"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "eval"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Project exists:", PROJECT_ROOT.exists())
print("Runs exists:", RUNS_ROOT.exists())
print("Validation data exists:", (DATA_ROOT / "val.csv").exists())

Project exists: True
Runs exists: True
Validation data exists: True


In [6]:
import os
import sys

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Working directory:", os.getcwd())

Working directory: /content/drive/MyDrive/2026-1/NLP/appraisal/nlp-appraisal


In [7]:
from src.config import TARGET_DIMS, OBJECTIVE_GROUPS
from src.dataset import AppraisalDataset
from src.eval import evaluate_model
from src.model import build_model

print("Project imports successful")
print("Number of appraisal dimensions:", len(TARGET_DIMS))

Project imports successful
Number of appraisal dimensions: 21


In [8]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader
from transformers import AutoTokenizer

from src.config import (
    BATCH_SIZE,
    DROPOUT_P,
    GROUP_HIDDEN_DIM,
    MAX_LENGTH,
    MODEL_NAME,
    OBJECTIVE_GROUPS,
    SHARED_DIM,
    TARGET_DIMS,
    TEXT_COLUMN,
)

from src.dataset import AppraisalDataset
from src.eval import evaluate_model
from src.model import build_model


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Evaluation device:", DEVICE)


def json_safe(value):
    if isinstance(value, dict):
        return {
            key: json_safe(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [
            json_safe(item)
            for item in value
        ]

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, np.generic):
        return value.item()

    return value


def evaluate_split_run(
    run_name: str,
    split: str = "val",
    batch_size: int = 16,
    overwrite: bool = False,
) -> dict:
    if split not in {"val", "test"}:
        raise ValueError(
            f"split must be 'val' or 'test', got {split!r}"
        )
    run_dir = RUNS_ROOT / run_name

    config_path = run_dir / "config.json"
    checkpoint_path = run_dir / "best_model.pt"
    split_path = DATA_ROOT / f"{split}.csv"

    metrics_output_path = (
        run_dir / f"{split}_metrics_recomputed.json"
    )

    predictions_output_path = (
        run_dir / f"{split}_predictions_recomputed.csv"
    )

    if (
        metrics_output_path.exists()
        and predictions_output_path.exists()
        and not overwrite
    ):
        print(f"Skipping completed run: {run_name}")

        with open(
            metrics_output_path,
            "r",
            encoding="utf-8",
        ) as file:
            return json.load(file)

    required_paths = [
        run_dir,
        config_path,
        checkpoint_path,
        split_path,
    ]

    for path in required_paths:
        if not path.exists():
            raise FileNotFoundError(path)

    with open(
        config_path,
        "r",
        encoding="utf-8",
    ) as file:
        run_config = json.load(file)

    model_type = run_config["model_type"]
    loss_mode = run_config["loss"]

    print()
    print("=" * 70)
    print("Run:", run_name)
    print("Architecture:", model_type)
    print("Loss:", loss_mode)
    print("Checkpoint:", checkpoint_path)
    print("=" * 70)

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME
    )

    weights_json = None
    weights_path = DATA_ROOT / "dim_weights.json"

    if loss_mode in {
        "weighted_mse",
        "weighted_group_balanced_mse",
    }:
        with open(
            weights_path,
            "r",
            encoding="utf-8",
        ) as file:
            weights_json = json.load(file)

    dataset = AppraisalDataset(
        csv_path=str(split_path),
        tokenizer=tokenizer,
        target_dims=TARGET_DIMS,
        text_column=TEXT_COLUMN,
        max_length=MAX_LENGTH,
        weights=weights_json,
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=(DEVICE == "cuda"),
    )

    model = build_model(
        model_type=model_type,
        model_name=MODEL_NAME,
        target_dims=TARGET_DIMS,
        objective_groups=OBJECTIVE_GROUPS,
        shared_dim=SHARED_DIM,
        group_hidden_dim=GROUP_HIDDEN_DIM,
        dropout_p=DROPOUT_P,
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False,
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    model.to(DEVICE)

    results = evaluate_model(
        model=model,
        dataloader=loader,
        target_dims=TARGET_DIMS,
        objective_groups=OBJECTIVE_GROUPS,
        loss_mode=loss_mode,
        device=DEVICE,
    )

    metrics = {
        key: value
        for key, value in results.items()
        if key not in {
            "all_predictions",
            "all_labels",
            "all_masks",
        }
    }

    with open(
        metrics_output_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            json_safe(metrics),
            file,
            indent=2,
        )

    split_df = pd.read_csv(split_path)

    predictions = results["all_predictions"]
    labels = results["all_labels"]
    masks = results["all_masks"]

    output_df = pd.DataFrame()

    if TEXT_COLUMN in split_df.columns:
        output_df[TEXT_COLUMN] = split_df[TEXT_COLUMN]

    for dim_idx, dim_name in enumerate(
        TARGET_DIMS
    ):
        output_df[f"{dim_name}_true"] = np.where(
            masks[:, dim_idx],
            labels[:, dim_idx],
            np.nan,
        )

        output_df[f"{dim_name}_pred"] = (
            predictions[:, dim_idx]
        )

    output_df.to_csv(
        predictions_output_path,
        index=False,
    )

    ranking = results["ranking_metrics"]
    high_intensity = results[
        "high_intensity_metrics"
    ]

    print(
        f"{split.title()} macro RMSE: "
        f"{results['macro_rmse']:.6f}"
    )

    print(
        f"{split.title()} macro MAE: "
        f"{results['macro_mae']:.6f}"
    )

    print(
        f"{split.title()} macro Pearson: "
        f"{results['macro_pearson']:.6f}"
    )

    print(
        f"Within-entry Spearman: "
        f"{ranking['mean_within_entry_spearman']:.6f}"
    )

    print(
        f"Top-3 overlap: "
        f"{ranking['top3_overlap']:.6f}"
    )

    print(
        f"High-intensity F1: "
        f"{high_intensity['micro_f1']:.6f}"
    )

    print("Saved:", metrics_output_path.name)
    print("Saved:", predictions_output_path.name)

    del model
    del checkpoint
    del loader
    del dataset

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return metrics

Evaluation device: cuda


In [9]:
RUNS_TO_RECOMPUTE = [
    "flat_linear_ft_mse_seed123",
    "flat_linear_ft_mse_seed456",
    "flat_mlp_ft_mse_seed123",
    "flat_mlp_ft_mse_seed456",
    "grouped_parallel_ft_mse_seed456",
    "grouped_sequential_ft_mse_seed42",
    "grouped_sequential_ft_mse_seed123",
    "grouped_sequential_ft_mse_seed456",
]

In [10]:
recomputed_results = {}

for run_name in RUNS_TO_RECOMPUTE:
    try:
        recomputed_results[run_name] = (
            evaluate_split_run(
                run_name=run_name,
                batch_size=16,
                overwrite=False,
            )
        )
    except Exception as error:
        print()
        print(f"FAILED: {run_name}")
        print(repr(error))
        break

Skipping completed run: flat_linear_ft_mse_seed123
Skipping completed run: flat_linear_ft_mse_seed456
Skipping completed run: flat_mlp_ft_mse_seed123
Skipping completed run: flat_mlp_ft_mse_seed456
Skipping completed run: grouped_parallel_ft_mse_seed456
Skipping completed run: grouped_sequential_ft_mse_seed42
Skipping completed run: grouped_sequential_ft_mse_seed123
Skipping completed run: grouped_sequential_ft_mse_seed456


In [11]:
for run_name in RUNS_TO_RECOMPUTE:
    run_dir = RUNS_ROOT / run_name

    metrics_path = (
        run_dir / "val_metrics_recomputed.json"
    )

    predictions_path = (
        run_dir / "val_predictions_recomputed.csv"
    )

    print(
        run_name,
        "| metrics:",
        metrics_path.exists(),
        "| predictions:",
        predictions_path.exists(),
    )

flat_linear_ft_mse_seed123 | metrics: True | predictions: True
flat_linear_ft_mse_seed456 | metrics: True | predictions: True
flat_mlp_ft_mse_seed123 | metrics: True | predictions: True
flat_mlp_ft_mse_seed456 | metrics: True | predictions: True
grouped_parallel_ft_mse_seed456 | metrics: True | predictions: True
grouped_sequential_ft_mse_seed42 | metrics: True | predictions: True
grouped_sequential_ft_mse_seed123 | metrics: True | predictions: True
grouped_sequential_ft_mse_seed456 | metrics: True | predictions: True


In [12]:
ALL_FINAL_RUNS = [
    "flat_linear_ft_mse_seed42",
    "flat_linear_ft_mse_seed123",
    "flat_linear_ft_mse_seed456",

    "flat_mlp_ft_mse_seed42",
    "flat_mlp_ft_mse_seed123",
    "flat_mlp_ft_mse_seed456",

    "grouped_parallel_ft_mse_seed42",
    "grouped_parallel_ft_mse_seed123",
    "grouped_parallel_ft_mse_seed456",

    "grouped_sequential_ft_mse_seed42",
    "grouped_sequential_ft_mse_seed123",
    "grouped_sequential_ft_mse_seed456",
]

In [13]:
for run_name in ALL_FINAL_RUNS:
    evaluate_split_run(
        run_name=run_name,
        batch_size=16,
        overwrite=False,
    )

Skipping completed run: flat_linear_ft_mse_seed42
Skipping completed run: flat_linear_ft_mse_seed123
Skipping completed run: flat_linear_ft_mse_seed456
Skipping completed run: flat_mlp_ft_mse_seed42
Skipping completed run: flat_mlp_ft_mse_seed123
Skipping completed run: flat_mlp_ft_mse_seed456
Skipping completed run: grouped_parallel_ft_mse_seed42
Skipping completed run: grouped_parallel_ft_mse_seed123
Skipping completed run: grouped_parallel_ft_mse_seed456
Skipping completed run: grouped_sequential_ft_mse_seed42
Skipping completed run: grouped_sequential_ft_mse_seed123
Skipping completed run: grouped_sequential_ft_mse_seed456


In [14]:
for run_name in ALL_FINAL_RUNS:
    run_dir = RUNS_ROOT / run_name

    metrics_path = run_dir / "val_metrics_recomputed.json"
    predictions_path = run_dir / "val_predictions_recomputed.csv"

    print(
        f"{run_name:45} "
        f"metrics={metrics_path.exists()} "
        f"predictions={predictions_path.exists()}"
    )

flat_linear_ft_mse_seed42                     metrics=True predictions=True
flat_linear_ft_mse_seed123                    metrics=True predictions=True
flat_linear_ft_mse_seed456                    metrics=True predictions=True
flat_mlp_ft_mse_seed42                        metrics=True predictions=True
flat_mlp_ft_mse_seed123                       metrics=True predictions=True
flat_mlp_ft_mse_seed456                       metrics=True predictions=True
grouped_parallel_ft_mse_seed42                metrics=True predictions=True
grouped_parallel_ft_mse_seed123               metrics=True predictions=True
grouped_parallel_ft_mse_seed456               metrics=True predictions=True
grouped_sequential_ft_mse_seed42              metrics=True predictions=True
grouped_sequential_ft_mse_seed123             metrics=True predictions=True
grouped_sequential_ft_mse_seed456             metrics=True predictions=True


In [15]:
%run analysis/compare_architectures_multiseed_validation.py


Per-seed validation results:
  Architecture  Seed  Validation Macro RMSE  Validation Macro MAE  Validation Macro Pearson  Within-entry Spearman  Top-3 Overlap  High-intensity F1
  CPM parallel    42               0.314412              0.251369                  0.532931               0.587959       0.357215           0.441922
  CPM parallel   123               0.314295              0.250746                  0.533763               0.586099       0.355869           0.435512
  CPM parallel   456               0.314658              0.251704                  0.533528               0.589420       0.359569           0.458384
CPM sequential    42               0.314743              0.247571                  0.536897               0.589093       0.366297           0.483642
CPM sequential   123               0.314070              0.250070                  0.538739               0.590798       0.364951           0.464369
CPM sequential   456               0.313216              0.248606           

In [16]:
import json

run_name = "grouped_sequential_ft_mse_seed42"
run_dir = RUNS_ROOT / run_name

with open(run_dir / "best_model.metrics.json") as f:
    original = json.load(f)

with open(run_dir / "val_metrics_recomputed.json") as f:
    recomputed = json.load(f)

for key in [
    "macro_rmse",
    "macro_mae",
    "macro_pearson",
]:
    print(
        key,
        "original=",
        original.get(key),
        "recomputed=",
        recomputed.get(key),
    )

macro_rmse original= 0.31474331872803823 recomputed= 0.31474332156635465
macro_mae original= 0.24757143003599985 recomputed= 0.24757143287431627
macro_pearson original= 0.5368969748895934 recomputed= 0.536896959716031


In [17]:
print(evaluate_split_run)

<function evaluate_split_run at 0x7f720972c900>


In [18]:
FINAL_TEST_RUNS = [
    "grouped_sequential_ft_mse_seed42",
    "grouped_sequential_ft_mse_seed123",
    "grouped_sequential_ft_mse_seed456",
]

for run_name in FINAL_TEST_RUNS:
    evaluate_split_run(
        run_name=run_name,
        split="test",
        batch_size=16,
        overwrite=True,
    )


Run: grouped_sequential_ft_mse_seed42
Architecture: grouped_sequential
Loss: mse
Checkpoint: /content/drive/MyDrive/2026-1/NLP/appraisal/nlp-appraisal/runs/grouped_sequential_ft_mse_seed42/best_model.pt


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

...

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Test macro RMSE: 0.316639
Test macro MAE: 0.251608
Test macro Pearson: 0.531592
Within-entry Spearman: 0.590631
Top-3 overlap: 0.362963
High-intensity F1: 0.470034
Saved: test_metrics_recomputed.json
Saved: test_predictions_recomputed.csv

Run: grouped_sequential_ft_mse_seed456
Architecture: grouped_sequential
Loss: mse
Checkpoint: /content/drive/MyDrive/2026-1/NLP/appraisal/nlp-appraisal/runs/grouped_sequential_ft_mse_seed456/best_model.pt


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Test macro RMSE: 0.318306
Test macro MAE: 0.251980
Test macro Pearson: 0.525222
Within-entry Spearman: 0.587218
Top-3 overlap: 0.361953
High-intensity F1: 0.474251
Saved: test_metrics_recomputed.json
Saved: test_predictions_recomputed.csv


In [19]:
FINAL_TEST_RUNS = [
    "grouped_sequential_ft_mse_seed42",
    "grouped_sequential_ft_mse_seed123",
    "grouped_sequential_ft_mse_seed456",
]

for run_name in FINAL_TEST_RUNS:
    run_dir = RUNS_ROOT / run_name

    metrics_path = run_dir / "test_metrics_recomputed.json"
    predictions_path = run_dir / "test_predictions_recomputed.csv"

    print(
        f"{run_name:45} "
        f"metrics={metrics_path.exists()} "
        f"predictions={predictions_path.exists()}"
    )

grouped_sequential_ft_mse_seed42              metrics=True predictions=True
grouped_sequential_ft_mse_seed123             metrics=True predictions=True
grouped_sequential_ft_mse_seed456             metrics=True predictions=True


In [20]:
import json
from pathlib import Path

FINAL_TEST_RUNS = [
    "grouped_sequential_ft_mse_seed42",
    "grouped_sequential_ft_mse_seed123",
    "grouped_sequential_ft_mse_seed456",
]

SEED_MAP = {
    "grouped_sequential_ft_mse_seed42": 42,
    "grouped_sequential_ft_mse_seed123": 123,
    "grouped_sequential_ft_mse_seed456": 456,
}

rows = []

for run_name in FINAL_TEST_RUNS:
    metrics_path = (
        RUNS_ROOT
        / run_name
        / "test_metrics_recomputed.json"
    )

    with open(
        metrics_path,
        "r",
        encoding="utf-8",
    ) as file:
        metrics = json.load(file)

    ranking = metrics["ranking_metrics"]
    high_intensity = metrics["high_intensity_metrics"]

    row = {
        "Run": run_name,
        "Seed": SEED_MAP[run_name],

        "Test Macro RMSE":
            metrics["macro_rmse"],

        "Test Macro MAE":
            metrics["macro_mae"],

        "Test Macro Pearson":
            metrics["macro_pearson"],

        "Within-entry Spearman":
            ranking["mean_within_entry_spearman"],

        "Exact Top-1 Accuracy":
            ranking["exact_top1_accuracy"],

        "Tie-aware Top-1 Accuracy":
            ranking["tie_aware_top1_accuracy"],

        "Top-3 Overlap":
            ranking["top3_overlap"],

        "Top-5 Overlap":
            ranking["top5_overlap"],

        "High-intensity Precision":
            high_intensity["micro_precision"],

        "High-intensity Recall":
            high_intensity["micro_recall"],

        "High-intensity F1":
            high_intensity["micro_f1"],
    }

    for group_name in [
        "relevance",
        "implication",
        "coping",
        "normative",
    ]:
        group_metrics = metrics[
            "group_metrics"
        ][group_name]

        row[f"{group_name.title()} RMSE"] = (
            group_metrics["mean_rmse"]
        )

        row[f"{group_name.title()} MAE"] = (
            group_metrics["mean_mae"]
        )

        row[f"{group_name.title()} Pearson"] = (
            group_metrics["mean_pearson"]
        )

    rows.append(row)


test_seed_df = pd.DataFrame(rows).sort_values("Seed")

metric_columns = [
    column
    for column in test_seed_df.columns
    if column not in {"Run", "Seed"}
]

summary_rows = []

for metric in metric_columns:
    values = pd.to_numeric(
        test_seed_df[metric],
        errors="coerce",
    ).dropna()

    summary_rows.append({
        "Metric": metric,
        "Mean": values.mean(),
        "SD": values.std(ddof=1),
        "Min": values.min(),
        "Max": values.max(),
    })

test_summary_df = pd.DataFrame(summary_rows)

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "eval"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

per_seed_path = (
    OUTPUT_DIR
    / "cpm_sequential_multiseed_test_by_seed.csv"
)

summary_path = (
    OUTPUT_DIR
    / "cpm_sequential_multiseed_test_summary.csv"
)

test_seed_df.to_csv(
    per_seed_path,
    index=False,
)

test_summary_df.to_csv(
    summary_path,
    index=False,
)

print("Per-seed test results:")
display(test_seed_df)

print("\nMulti-seed test summary:")
display(test_summary_df)

print("Saved:", per_seed_path)
print("Saved:", summary_path)

Per-seed test results:


,Run,Seed,Test Macro RMSE,Test Macro MAE,Test Macro Pearson,Within-entry Spearman,Exact Top-1 Accuracy,Tie-aware Top-1 Accuracy,Top-3 Overlap,Top-5 Overlap,...,Relevance Pearson,Implication RMSE,Implication MAE,Implication Pearson,Coping RMSE,Coping MAE,Coping Pearson,Normative RMSE,Normative MAE,Normative Pearson
0,grouped_sequential_ft_mse_seed42,42,0.318827,0.249741,0.526409,0.586303,0.245455,0.702020,0.360606,0.496566,...,0.558614,0.324635,0.259849,0.520902,0.330721,0.268322,0.449263,0.296807,0.203204,0.606978
1,grouped_sequential_ft_mse_seed123,123,0.316639,0.251608,0.531592,0.590631,0.231313,0.713131,0.362963,0.501010,...,0.562703,0.324200,0.262429,0.524704,0.329472,0.270914,0.454278,0.290108,0.205033,0.621099
2,grouped_sequential_ft_mse_seed456,456,0.318306,0.251980,0.525222,0.587218,0.270707,0.696970,0.361953,0.496566,...,0.561473,0.324544,0.262178,0.519480,0.331579,0.272720,0.441196,0.295291,0.204970,0.607510



Multi-seed test summary:


,Metric,Mean,SD,Min,Max
0,Test Macro RMSE,0.317924,0.001143,0.316639,0.318827
1,Test Macro MAE,0.251110,0.001200,0.249741,0.251980
2,Test Macro Pearson,0.527741,0.003388,0.525222,0.531592
3,Within-entry Spearman,0.588051,0.002281,0.586303,0.590631
4,Exact Top-1 Accuracy,0.249158,0.019956,0.231313,0.270707
5,Tie-aware Top-1 Accuracy,0.704040,0.008268,0.696970,0.713131
6,Top-3 Overlap,0.361841,0.001182,0.360606,0.362963
7,Top-5 Overlap,0.498047,0.002566,0.496566,0.501010
8,High-intensity Precision,0.800047,0.007713,0.793484,0.808542
9,High-intensity Recall,0.341460,0.012698,0.331322,0.355703


Saved: /content/drive/MyDrive/2026-1/NLP/appraisal/nlp-appraisal/outputs/eval/cpm_sequential_multiseed_test_by_seed.csv
Saved: /content/drive/MyDrive/2026-1/NLP/appraisal/nlp-appraisal/outputs/eval/cpm_sequential_multiseed_test_summary.csv


In [21]:
compact_metrics = [
    "Test Macro RMSE",
    "Test Macro MAE",
    "Test Macro Pearson",
    "Within-entry Spearman",
    "Top-3 Overlap",
    "High-intensity F1",
]

compact_summary = (
    test_summary_df[
        test_summary_df["Metric"].isin(compact_metrics)
    ]
    .copy()
)

compact_summary["Mean ± SD"] = compact_summary.apply(
    lambda row: (
        f"{row['Mean']:.4f} ± {row['SD']:.4f}"
    ),
    axis=1,
)

display(
    compact_summary[
        ["Metric", "Mean ± SD"]
    ]
)

,Metric,Mean ± SD
0,Test Macro RMSE,0.3179 ± 0.0011
1,Test Macro MAE,0.2511 ± 0.0012
2,Test Macro Pearson,0.5277 ± 0.0034
3,Within-entry Spearman,0.5881 ± 0.0023
6,Top-3 Overlap,0.3618 ± 0.0012
10,High-intensity F1,0.4785 ± 0.0112


In [22]:
macro_rmse_row = test_summary_df[
    test_summary_df["Metric"] == "Test Macro RMSE"
].iloc[0]

normalized_rmse_mean = macro_rmse_row["Mean"]
normalized_rmse_sd = macro_rmse_row["SD"]

original_scale_rmse_mean = normalized_rmse_mean * 4
original_scale_rmse_sd = normalized_rmse_sd * 4

print(
    "Original-scale RMSE:",
    f"{original_scale_rmse_mean:.4f} "
    f"± {original_scale_rmse_sd:.4f}"
)

Original-scale RMSE: 1.2717 ± 0.0046


In [ ]:
%run analysis/paired_bootstrap_validation.py

Loaded ensemble predictions: Flat linear
Loaded ensemble predictions: Flat MLP
Loaded ensemble predictions: CPM parallel
Loaded ensemble predictions: CPM sequential

CPM sequential vs Flat linear
Macro RMSE                diff=+0.000689 CI=[-0.000930, +0.002339] P(A better)=0.793
Macro MAE                 diff=+0.003308 CI=[+0.001911, +0.004726] P(A better)=1.000
Macro Pearson             diff=+0.006298 CI=[+0.000632, +0.011966] P(A better)=0.986


In [ ]:
%run analysis/paired_bootstrap_validation_by_seed.py

In [ ]:
%run analysis/analyze_final_test_dimensions.py

In [ ]:
%run analysis/plot_final_test_dimensions.py

In [ ]:
%run analysis/analyze_prediction_bias.py